##Installations

Run this block to install the required libraries for processing, vectorization, and generation.

In [1]:
# had to download older langchain because ragas still uses the olde langchain-community
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-core==0.2.43 langchain-huggingface==0.0.3 langchain-ollama==0.1.3 pypdf chromadb sentence-transformers deepeval pandas ragas datasets streamlit rank_bm25

## Multi-Format Document Processing & Database Precomputation
This block loads the raw documents into memory exactly once, and then iterates through different chunk sizes (500, 1000, 2000) to build three separate ChromaDB vector stores on the local disk.

In [2]:
import os
import glob
import json
import warnings
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader, CSVLoader # langchain is restructuring (splitting this package into smaller ones) so this might has a warning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/RAG_Project"
KB_PATH = f"{DRIVE_PROJECT_PATH}/knowledge_base/*"

# Load Raw Documents Once
def load_raw_documents(kb_folder=KB_PATH):
    print("Loading raw documents into memory...")
    all_files = glob.glob(kb_folder, recursive=True)
    raw_docs = []

    for file_path in all_files:
        if not os.path.isfile(file_path):
            continue

        file_name = os.path.basename(file_path)
        try:
            if file_path.endswith('.pdf'):
                raw_docs.extend(PyPDFLoader(file_path).load())
            elif file_path.endswith('.csv'):
                raw_docs.extend(CSVLoader(file_path).load())
            elif file_path.endswith('.json'):
                with open(file_path, 'r', encoding='utf-8') as f:
                    text_content = json.dumps(json.load(f), indent=2)
                    raw_docs.append(Document(page_content=text_content, metadata={"source": file_name}))
        except Exception as e:
            print(f"Error loading {file_name}: {e}")

    print(f"Loaded {len(raw_docs)} raw document pages.")
    return raw_docs

raw_documents = load_raw_documents()

# Setup Embedding Model
# warning might appear because the saved model weights include static 'position_ids', but the modern
# transformers library generates them dynamically at runtime, creating a safe mismatch.
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Chunk and Build Databases for Multiple Sizes
chunk_sizes = [500, 1000, 2000]
vector_stores = {}

for size in chunk_sizes:
    print(f"\n--- Processing Chunk Size: {size} ---")
    overlap = int(size * 0.15)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)

    chunks = text_splitter.split_documents(raw_documents)
    print(f"Generated {len(chunks)} chunks.")

    persist_dir = f"{DRIVE_PROJECT_PATH}/chroma_db_{size}"
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_dir
    )
    vector_stores[size] = vector_store
    print(f"ChromaDB saved to {persist_dir}")

# Set the 1000-chunk database as the baseline for downstream evaluation blocks
vector_store_baseline = vector_stores[1000]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading raw documents into memory...
Loaded 413 raw document pages.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Processing Chunk Size: 500 ---
Generated 2382 chunks.
ChromaDB saved to /content/drive/MyDrive/RAG_Project/chroma_db_500

--- Processing Chunk Size: 1000 ---
Generated 1281 chunks.
ChromaDB saved to /content/drive/MyDrive/RAG_Project/chroma_db_1000

--- Processing Chunk Size: 2000 ---
Generated 732 chunks.
ChromaDB saved to /content/drive/MyDrive/RAG_Project/chroma_db_2000


## Advanced RAG Pipeline Construction
Build the dynamic retrieval architecture featuring Naive Vector Search, BM25 Keyword Hybrid Search, and Cross-Encoder Reranking.

In [3]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.documents import Document

# Initialize Models
llm = OllamaLLM(model="llama3")
reranker_model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# Build Offline BM25 Index directly from ChromaDB chunks
print("Extracting chunks for BM25 Index...")
db_data = vector_store_baseline.get()
docs = [Document(page_content=txt, metadata=meta) for txt, meta in zip(db_data['documents'], db_data['metadatas'])]
bm25_retriever_base = BM25Retriever.from_documents(docs)
print("BM25 Index Built!")

# Dynamic Pipeline Builder
def build_rag_chain(strategy="Naive RAG", top_k=3):
    if strategy == "Naive RAG":
        retriever = vector_store_baseline.as_retriever(search_kwargs={"k": top_k})

    elif strategy == "Hybrid Search":
        chroma_retriever = vector_store_baseline.as_retriever(search_kwargs={"k": top_k})
        bm25_retriever_base.k = top_k
        retriever = EnsembleRetriever(retrievers=[bm25_retriever_base, chroma_retriever], weights=[0.5, 0.5])

    elif strategy == "Hybrid + Reranker":
        initial_k = top_k * 3 # Fetch more for the reranker to sort
        chroma_retriever = vector_store_baseline.as_retriever(search_kwargs={"k": initial_k})
        bm25_retriever_base.k = initial_k
        ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever_base, chroma_retriever], weights=[0.5, 0.5])
        compressor = CrossEncoderReranker(model=reranker_model, top_n=top_k)
        retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=ensemble_retriever)

    system_prompt = (
        "You are an expert incident response practitioner. "
        "Use the following retrieved context to generate 3 to 5 concise guidelines. "
        "CRITICAL RULE 1: You MUST cite the exact source document name (e.g., [filename.pdf]) for every step. Do not just use numbers like [1]. "
        "CRITICAL RULE 2: Do NOT recommend expensive enterprise tools like SIEMs or assume the user has a massive SOC team. Keep advice universally applicable for low-resource environments. "
        "Context:\n{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    return create_retrieval_chain(retriever, question_answer_chain)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting chunks for BM25 Index...
BM25 Index Built!


## Local Server Initialization
Boot up the local Ollama server in the background and pull the Llama 3 weights.

In [4]:
# Install missing dependencies (zstd for extraction, pciutils for GPU)
!sudo apt install -y zstd pciutils > /dev/null

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start the server completely detached from the Colab cell
!nohup ollama serve > ollama_server.log 2>&1 </dev/null &

# Give the server a few seconds to wake up
!sleep 5

# Download the Llama 3 model
!ollama pull llama3



>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



## Automated Evaluation (DeepEval)
Runs the evaluation queries through the baseline pipeline and scores the outputs using local custom metrics.

In [5]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import OllamaModel
from deepeval import evaluate

# Change this to "Naive RAG", "Hybrid Search", or "Hybrid + Reranker" depends on that type of RAG to test
TARGET_STRATEGY = "Hybrid + Reranker"
active_chain = build_rag_chain(strategy=TARGET_STRATEGY, top_k=3)

print(f"Connecting to local Llama 3 server to evaluate: {TARGET_STRATEGY}...")

local_judge_model = OllamaModel(model="llama3", base_url="http://localhost:11434", temperature=0)

inclusivity_metric = GEval(
    name="Inclusivity and Resource Bias",
    criteria="Determine if the generated guidelines possess a resource bias (e.g., assuming access to expensive enterprise tools). Explicitly evaluate if the advice is practical and inclusive for indigenous, small, or under-resourced environments.",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=local_judge_model,
)

transparency_metric = GEval(
    name="Source Transparency",
    criteria="Determine whether the actual output explicitly cites the source documents provided in the retrieval context.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT],
    model=local_judge_model,
)

actionability_metric = GEval(
    name="Actionability",
    criteria="Determine if the generated guidelines are practical, clear, and immediately actionable for an incident responder, rather than vague high-level concepts.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=local_judge_model,
)

eval_queries = [
    "What are the immediate containment steps for a compromised cloud server?",
    "How do we safely capture volatile memory during an active breach?",
    "What port does TruffleHog run on?"
]

deep_eval_test_cases = []
print(f"Running evaluation queries through {TARGET_STRATEGY}...")

for query in eval_queries:
    response = active_chain.invoke({"input": query})

    test_case = LLMTestCase(
        input=query,
        actual_output=response["answer"],
        retrieval_context=[doc.page_content for doc in response["context"]]
    )
    deep_eval_test_cases.append(test_case)

deepeval_results = evaluate(deep_eval_test_cases, metrics=[inclusivity_metric, transparency_metric, actionability_metric])

print(deepeval_results)

Connecting to local Llama 3 server to evaluate: Hybrid + Reranker...
Running evaluation queries through Hybrid + Reranker...


✨ You're running DeepEval's latest Inclusivity and Resource Bias [GEval] Metric! (using llama3 (Ollama), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Source Transparency [GEval] Metric! (using llama3 (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Actionability [GEval] Metric! (using llama3 (Ollama), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            What are the immediate containment steps for a compromised cloud server?               │
│  │     Actual Output:    Based on the provided context, here are three to five concise guidelines for           │
│  │                       immediate containment steps for a compromised cloud server:                            │
│  │                                                                                                              │
│  │                       [Contingency_Plan_Template.docx]                                                       │
│  │                                                                                                              │
│  │                       1. **Isolate the affected cloud server**: Immediately isolate the compromised cloud    │
│  │                       server from the rest of the network and other servers to prevent further lateral       │
│  │                       movement or data exfiltration. This can be done by configuring network ACLs,           │
│  │                       firewall rules, or isolating the server in a quarantine network segment (Bejtlich,     │
│  │                       2005).                                                                                 │
│  │                                                                                                              │
│  │                       [Incident_Response_Handbook_v1.0.pdf]                                                  │
│  │                                                                                                              │
│  │                       2. **Disconnect power and internet connectivity**: Disconnect power to the             │
│  │                       compromised cloud server and remove its internet connection to prevent any             │
│  │                       malicious activity from continuing. This is especially important for servers that      │
│  │                       have been hacked by an adversary using fileless malware (Uf it security, 2011).        │
│  │                                                                                                              │
│  │                       [Cloud_Security_Guidelines_v2.0.pdf]                                                   │
│  │                                                                                                              │
│  │                       3. **Configure logging and monitoring**: Configure logging and monitoring tools to     │
│  │                       collect relevant data about the compromised cloud server's activities before and       │
│  │                       after containment. This will help investigators reconstruct what happened during       │
│  │                       the incident and identify potential indicators of compromise (IOC) for future          │
│  │                       detection.                                                                             │
│  │                                                                                                              │
│  │                       [Incident_Response_Playbook_v1.2.docx]                                                 │
│  │                                                      

⚠ WARNING: No hyperparameters logged.
» ]8;id=251778;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 68.7s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_results=[TestResult(name='test_case_1', success=False, metrics_data=[MetricData(name='Inclusivity and Resource Bias [GEval]', threshold=0.5, success=False, score=0.4, reason='The guidelines prioritize accessibility and inclusivity for under-resourced environments by providing adaptable solutions, such as using trusted media or forensic disk imaging tools. However, the assumption of access to incident_response_procedure.pdf may not be practical for all environments. Additionally, the containment and isolation steps require specialized equipment or expertise, which could be a limitation.', strict_mode=False, evaluation_model='llama3 (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Criteria:\nDetermine if the generated guidelines possess a resource bias (e.g., assuming access to expensive enterprise tools). Explicitly evaluate if the advice is practical and inclusive for indigenous, small, or under-resourced environments. \n \nEvaluation Steps:\n[\n    "Evaluate Input: Check

## Extended Evaluation: RAGAS Metrics
Further evaluates the test cases for faithfulness and answer relevancy, and build an overall evaluation dataframe.

In [6]:
from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import answer_relevancy, faithfulness
# ragas is being reorganize and changing to newer version of ragas metrics need too much change to the pipeline

print("Formatting data for RAGAS...")

# Extract the data already generated during the DeepEval loop
ragas_data = {
    "question": [tc.input for tc in deep_eval_test_cases],
    "answer": [tc.actual_output for tc in deep_eval_test_cases],
    "contexts": [tc.retrieval_context for tc in deep_eval_test_cases],
}

dataset = Dataset.from_dict(ragas_data)

print("Running RAGAS metrics...")

# Evaluate using the same local LLM and Embeddings defined in Block 4
ragas_results = ragas_evaluate(
    dataset=dataset,
    metrics=[answer_relevancy, faithfulness],
    llm=llm,
    embeddings=embedding_model,
)
df_eval = ragas_results.to_pandas()

# Extract DeepEval Scores and Reasons
inclusivity_scores, inclusivity_reasons = [], []
transparency_scores, transparency_reasons = [], []
actionability_scores, actionability_reasons = [], []

# There is a problem where turning deepeval_results into a list does not work, this block makes it consistent
if hasattr(deepeval_results, 'test_results'):
    actual_results = deepeval_results.test_results
elif isinstance(deepeval_results, list):
    actual_results = deepeval_results
else:
    try:
        # Cast Pydantic object to dict and grab the specific key
        actual_results = dict(deepeval_results).get('test_results', deepeval_results)
    except Exception:
        actual_results = deepeval_results

# Loop through test cases
for result in actual_results:
    case_metrics = {}
    for m in result.metrics_data:
        if "Inclusivity" in m.name:
            case_metrics["inclusivity"] = (m.score, m.reason)
        elif "Transparency" in m.name:
            case_metrics["transparency"] = (m.score, m.reason)
        elif "Actionability" in m.name:
            case_metrics["actionability"] = (m.score, m.reason)

    # Append values (0.0 and "N/A" if empty)
    inc = case_metrics.get("inclusivity", (0.0, "N/A"))
    inclusivity_scores.append(inc[0])
    inclusivity_reasons.append(inc[1])

    trans = case_metrics.get("transparency", (0.0, "N/A"))
    transparency_scores.append(trans[0])
    transparency_reasons.append(trans[1])

    act = case_metrics.get("actionability", (0.0, "N/A"))
    actionability_scores.append(act[0])
    actionability_reasons.append(act[1])

# Append everything to the DataFrame
df_eval["deepeval_inclusivity_score"] = inclusivity_scores
df_eval["deepeval_inclusivity_reason"] = inclusivity_reasons

df_eval["deepeval_transparency_score"] = transparency_scores
df_eval["deepeval_transparency_reason"] = transparency_reasons

df_eval["deepeval_actionability_score"] = actionability_scores
df_eval["deepeval_actionability_reason"] = actionability_reasons

# Display the evaluation table
display(df_eval)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_70225/1130059895.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_relevancy, faithfulness
/tmp/ipykernel_70225/1130059895.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfuln

Formatting data for RAGAS...
Running RAGAS metrics...


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,answer_relevancy,faithfulness,deepeval_inclusivity_score,deepeval_inclusivity_reason,deepeval_transparency_score,deepeval_transparency_reason,deepeval_actionability_score,deepeval_actionability_reason
0,What are the immediate containment steps for a...,"[from happening (""Uf it security,"" 2011). Ther...","Based on the provided context, here are three ...",1.000000,0.333333,0.4,The guidelines prioritize accessibility and in...,0.2,The actual output does not explicitly cite the...,0.7,"The guidelines are specific, actionable, and c..."
1,How do we safely capture volatile memory durin...,[develop procedures based on those discussions...,"Based on the provided context, here are three ...",0.963088,0.285714,0.0,The response does not consider the limitations...,0.2,The actual output does not explicitly cite the...,0.7,"The guidelines are specific, actionable, and c..."
2,What port does TruffleHog run on?,"[in terms of their own policies and practices,...","Based on the provided context, I will generate...",0.369868,0.411765,0.8,The response does not assume access to expensi...,0.2,The actual output does not explicitly cite the...,0.8,"The guidelines are specific, actionable, and c..."


## 7. Streamlit Application Script
Writes the interactive dashboard logic (with chunk size toggling) to a local `app.py` file.

In [7]:
%%writefile app.py
import streamlit as st
from langchain_community.vectorstores import Chroma
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.documents import Document

# --- UI CONFIGURATION ---
st.set_page_config(page_title="RAG Dashboard", layout="wide")
st.title("Incident Response RAG Assistant")

# --- CACHED RESOURCES ---
@st.cache_resource
def load_models():
    llm = Ollama(model="llama3", base_url="http://localhost:11434", temperature=0)
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    # Load the Cross-Encoder for Reranking (Lightweight & Offline)
    reranker_model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
    return llm, embeddings, reranker_model

llm, embeddings, reranker_model = load_models()

@st.cache_resource
def get_vectorstore(chunk_size):
    drive_path = f"/content/drive/MyDrive/RAG_Project/chroma_db_{chunk_size}"
    return Chroma(persist_directory=drive_path, embedding_function=embeddings)

@st.cache_resource
def get_bm25_retriever(chunk_size):
    # Extract existing chunks from ChromaDB to build the BM25 exact-keyword index
    vector_store = get_vectorstore(chunk_size)
    db_data = vector_store.get()
    docs = [Document(page_content=txt, metadata=meta) for txt, meta in zip(db_data['documents'], db_data['metadatas'])]
    return BM25Retriever.from_documents(docs)

# --- SIDEBAR CONTROLS ---
st.sidebar.header("Pipeline Settings")
strategy = st.sidebar.radio("Retrieval Strategy", ["Naive RAG", "Hybrid Search", "Hybrid + Reranker"])
selected_chunk_size = st.sidebar.selectbox("Chunk Size", [500, 1000, 2000], index=1)
top_k = st.sidebar.slider("Final Top-K Chunks", min_value=1, max_value=10, value=3)

vector_store = get_vectorstore(selected_chunk_size)

# --- RETRIEVER LOGIC ---
if strategy == "Naive RAG":
    retriever = vector_store.as_retriever(search_kwargs={"k": top_k})

elif strategy == "Hybrid Search":
    chroma_retriever = vector_store.as_retriever(search_kwargs={"k": top_k})
    bm25_retriever = get_bm25_retriever(selected_chunk_size)
    bm25_retriever.k = top_k
    # 50% Vector Semantic Search / 50% Keyword Search
    retriever = EnsembleRetriever(retrievers=[bm25_retriever, chroma_retriever], weights=[0.5, 0.5])

elif strategy == "Hybrid + Reranker":
    # Fetch 3x more documents initially so the Reranker has options to sort through
    initial_k = top_k * 3
    chroma_retriever = vector_store.as_retriever(search_kwargs={"k": initial_k})
    bm25_retriever = get_bm25_retriever(selected_chunk_size)
    bm25_retriever.k = initial_k
    ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, chroma_retriever], weights=[0.5, 0.5])

    # Compress/Rerank down to the final Top-K
    compressor = CrossEncoderReranker(model=reranker_model, top_n=top_k)
    retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=ensemble_retriever)


# --- MAIN CHAT INTERFACE ---
st.subheader(f"Chat with the Knowledge Base ({strategy})")
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if "context" in msg and msg["context"]:
            with st.expander("🔍 View Retrieved Context Chunks"):
                for i, doc in enumerate(msg["context"]):
                    st.markdown(f"**Chunk {i+1} | Source: `{doc.metadata.get('source', 'Unknown')}`**")
                    st.caption(doc.page_content)
                    st.divider()

if prompt := st.chat_input("Enter your incident response query..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner(f"Running {strategy}..."):
            system_prompt = (
                "You are an expert incident response practitioner. "
                "Use the following retrieved context to answer the query concisely. "
                "CRITICAL RULE: You MUST cite the exact source document name (e.g., [filename.pdf]). "
                "Context:\n{context}"
            )
            qa_prompt = ChatPromptTemplate.from_messages([
                ("system", system_prompt),
                ("human", "{input}")
            ])
            question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
            rag_chain = create_retrieval_chain(retriever, question_answer_chain)

            response = rag_chain.invoke({"input": prompt})
            answer = response["answer"]
            context_chunks = response.get("context", [])

            st.markdown(answer)

            if not context_chunks:
                st.warning("⚠️ No context chunks retrieved.")
            else:
                with st.expander("🔍 View Retrieved Context Chunks"):
                    for i, doc in enumerate(context_chunks):
                        st.markdown(f"**Chunk {i+1} | Source: `{doc.metadata.get('source', 'Unknown')}`**")
                        st.caption(doc.page_content)
                        st.divider()

    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
        "context": context_chunks
    })

Overwriting app.py


## Deployment: Secure Tunneling & UI Hosting
Launch the Streamlit server in the background and expose it via a Cloudflare Quick Tunnel for live demonstration.

In [9]:
# Download Cloudflare's tunneling tool
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start Streamlit in the background
!nohup streamlit run app.py \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > streamlit.log 2>&1 &

# Open the Cloudflare tunnel
import time
print("Starting Cloudflare Tunnel...")
time.sleep(3) # Give Streamlit a second to boot
!./cloudflared tunnel --url http://localhost:8501

Starting Cloudflare Tunnel...
2026-05-25T13:42:24Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-25T13:42:24Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-25T13:42:29Z INF +--------------------------------------------------------------------------------------------+
2026-05-25T13:42:29Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-25T13:42:29Z INF |  https://prozac-mirrors-